# 🚇 BART Real-Time Feed - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

This notebook pulls BART's **live** GTFS-Realtime feeds and indexes upcoming train arrivals (with delays) and service alerts into Elasticsearch - a snapshot of the system *right now*.

**What this notebook does:**
1. Downloads BART's static GTFS so stop and route IDs become station and line names
2. Fetches the live trip-update and alert feeds from the [BART API](https://www.bart.gov/schedules/developers/gtfs-realtime) - **no API key required**
3. Indexes arrivals into `bart_realtime` and alerts into `bart_alerts`
4. **(Optional)** does the same for **Muni** via 511.org (free token)

> ⚠️ This is **live, ephemeral** data - each pull is a snapshot. Re-run to refresh.

---
> **Data source:** BART GTFS and GTFS-Realtime (public, no key). Muni real-time comes from [511.org Open Data](https://511.org/open-data/transit), which needs a free API token.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests gtfs-realtime-bindings protobuf --quiet

In [ ]:
import io, csv, zipfile, requests
from datetime import datetime, timezone
from google.transit import gtfs_realtime_pb2
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Load the static schedule (for names)
The live feed only carries IDs like trip `1973427` and stop `M20-1`. The static GTFS turns those into a line name and a station like "Montgomery Street".

In [ ]:
zf = zipfile.ZipFile(io.BytesIO(requests.get('https://www.bart.gov/dev/schedules/google_transit.zip', timeout=120).content))

def read_csv(name):
    with zf.open(name) as f:
        return list(csv.DictReader(io.TextIOWrapper(f, encoding='utf-8-sig')))

stops  = {s['stop_id']: s for s in read_csv('stops.txt')}
routes = {r['route_id']: r for r in read_csv('routes.txt')}
trips  = {t['trip_id']: t for t in read_csv('trips.txt')}
print(f'✅ {len(stops)} stops, {len(routes)} routes, {len(trips)} scheduled trips')

## 3. Fetch and parse the live trip updates

In [ ]:
def parse_feed(url, params=None):
    feed = gtfs_realtime_pb2.FeedMessage()
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    feed.ParseFromString(r.content)
    return feed

def station_of(stop_id):
    # Platform IDs like 'M20-1' belong to a parent station ('MONT' = Montgomery Street)
    s = stops.get(stop_id) or {}
    parent = stops.get(s.get('parent_station')) or s
    return parent

fetched_at = datetime.now(tz=timezone.utc).isoformat()
feed = parse_feed('https://api.bart.gov/gtfsrt/tripupdate.aspx')

arrivals = []
for entity in feed.entity:
    if not entity.HasField('trip_update'):
        continue
    tu = entity.trip_update
    trip = trips.get(tu.trip.trip_id, {})
    route = routes.get(tu.trip.route_id or trip.get('route_id'), {})
    for stu in tu.stop_time_update:
        if not stu.HasField('arrival') or not stu.arrival.time:
            continue
        st = station_of(stu.stop_id)
        doc = {
            'trip_id':      tu.trip.trip_id,
            'route_id':     route.get('route_id'),
            'line':         route.get('route_long_name') or route.get('route_short_name'),
            'line_color':   route.get('route_color'),
            'headsign':     trip.get('trip_headsign'),
            'stop_id':      stu.stop_id,
            'station':      st.get('stop_name'),
            'arrival_time': datetime.fromtimestamp(stu.arrival.time, tz=timezone.utc).isoformat(),
            'delay_sec':    stu.arrival.delay,
            'fetched_at':   fetched_at,
        }
        if st.get('stop_lat'):
            doc['location'] = {'lat': float(st['stop_lat']), 'lon': float(st['stop_lon'])}
        arrivals.append({k: v for k, v in doc.items() if v not in (None, '')})

print(f'✅ Parsed {len(arrivals)} arrival predictions from {len(feed.entity)} trips')

## 4. Create indices and ingest the snapshot (+ alerts)

In [ ]:
INDEX = 'bart_realtime'
if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
es.indices.create(index=INDEX, body={'mappings': {'properties': {
    'trip_id':      {'type': 'keyword'},
    'route_id':     {'type': 'keyword'},
    'line':         {'type': 'keyword'},
    'line_color':   {'type': 'keyword'},
    'headsign':     {'type': 'keyword'},
    'stop_id':      {'type': 'keyword'},
    'station':      {'type': 'keyword'},
    'arrival_time': {'type': 'date'},
    'delay_sec':    {'type': 'integer'},
    'fetched_at':   {'type': 'date'},
    'location':     {'type': 'geo_point'},
}}})
helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in arrivals], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {len(arrivals)} arrivals into {INDEX}')

# Service alerts: plain text, perfect for a Mistral summary
ALERTS_INDEX = 'bart_alerts'
alerts = []
for entity in parse_feed('https://api.bart.gov/gtfsrt/alerts.aspx').entity:
    if entity.HasField('alert'):
        a = entity.alert
        english = lambda ts: ' '.join(t.text for t in ts.translation if t.language.startswith('en') or not t.language)
        text, header = english(a.description_text), english(a.header_text)
        alerts.append({'alert_id': entity.id, 'header': header, 'description': text, 'fetched_at': fetched_at})
if es.indices.exists(index=ALERTS_INDEX):
    es.indices.delete(index=ALERTS_INDEX)
es.indices.create(index=ALERTS_INDEX, body={'mappings': {'properties': {
    'alert_id': {'type': 'keyword'}, 'header': {'type': 'text'},
    'description': {'type': 'text'}, 'fetched_at': {'type': 'date'},
}}})
helpers.bulk(es, [{'_index': ALERTS_INDEX, '_source': d} for d in alerts], raise_on_error=False)
es.indices.refresh(index=ALERTS_INDEX)
print(f'✅ Indexed {len(alerts)} alerts into {ALERTS_INDEX}')
for a in alerts[:3]:
    print(f"   • {a['description'][:120]}")

## 5. Verify

In [ ]:
resp = es.search(index=INDEX, body={'size': 0, 'aggs': {
    'by_line': {'terms': {'field': 'line', 'size': 10}, 'aggs': {'avg_delay': {'avg': {'field': 'delay_sec'}}}},
}})
print(f"Total arrival predictions: {resp['hits']['total']['value']}\n")
print('Upcoming arrivals and average delay by line:')
for b in resp['aggregations']['by_line']['buckets']:
    print(f"  {b['key'][:44]:44s} {b['doc_count']:4d}  avg delay {b['avg_delay']['value'] or 0:5.0f} s")

# Next trains at Montgomery Street (the closest station to the venue)
resp = es.search(index=INDEX, body={'size': 6,
    'query': {'term': {'station': 'Montgomery Street'}},
    'sort': [{'arrival_time': 'asc'}],
})
print('\nNext trains at Montgomery Street:')
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s['arrival_time'][11:16]} UTC  → {s.get('headsign', '?'):28s} (delay {s.get('delay_sec', 0)} s)")

## 6. (Optional) Muni real-time via 511.org
Muni's live predictions are published through **511.org**, which needs a free token: request one at [511.org/open-data/token](https://511.org/open-data/token) (it arrives by email in a minute or two). The feed is the same GTFS-Realtime format, so the parsing is identical. Join `stop_id` with `muni_stops` from [muni_schedule.ipynb](muni_schedule.ipynb) for names.

> 511 tokens are rate-limited (60 requests/hour by default) - poll gently.

In [ ]:
TOKEN_511 = '<YOUR_511_TOKEN>'   # 🔑 free from 511.org/open-data/token

if TOKEN_511.startswith('<'):
    print('⏭️  No 511 token set - skipping Muni real-time.')
else:
    feed = parse_feed('https://api.511.org/transit/tripupdates', params={'api_key': TOKEN_511, 'agency': 'SF'})
    muni = []
    for entity in feed.entity:
        if not entity.HasField('trip_update'):
            continue
        tu = entity.trip_update
        for stu in tu.stop_time_update:
            if stu.HasField('arrival') and stu.arrival.time:
                muni.append({'route_id': tu.trip.route_id, 'trip_id': tu.trip.trip_id, 'stop_id': stu.stop_id,
                             'arrival_time': datetime.fromtimestamp(stu.arrival.time, tz=timezone.utc).isoformat(),
                             'fetched_at': fetched_at})
    MUNI_INDEX = 'muni_realtime'
    if es.indices.exists(index=MUNI_INDEX):
        es.indices.delete(index=MUNI_INDEX)
    es.indices.create(index=MUNI_INDEX, body={'mappings': {'properties': {
        'route_id': {'type': 'keyword'}, 'trip_id': {'type': 'keyword'}, 'stop_id': {'type': 'keyword'},
        'arrival_time': {'type': 'date'}, 'fetched_at': {'type': 'date'},
    }}})
    helpers.bulk(es, [{'_index': MUNI_INDEX, '_source': d} for d in muni], raise_on_error=False)
    es.indices.refresh(index=MUNI_INDEX)
    print(f'✅ Indexed {len(muni)} Muni arrival predictions into {MUNI_INDEX}')

## 7. Ideas: handling live data (try these yourself!)

This notebook loads a one-time snapshot. Some more idiomatic ways to handle a live feed in Elastic:

1. **Deterministic `_id` with upserts** - use `_id = f"{trip_id}:{stop_id}"` so each poll overwrites the same documents in place. Stale trips age out via a `fetched_at` range filter at query time, or a periodic `delete_by_query`.
2. **Time-series data stream** - index each poll as append-only events with `@timestamp` = fetch time. You get delay history for free: "how did this train's predicted arrival drift over the last 10 minutes?" is a great demo story.
3. **A real poller** - run the fetch as a small script or scheduled job instead of a notebook cell.

## 8. Going further with Mistral

- **"Am I going to make it?"** A Mistral agent with a function that queries `bart_realtime`, answering "Will I make the 6:40 to the airport from Montgomery Street?"
- **Alerts in plain language:** have a Mistral model rewrite `bart_alerts` into a one-line status in the rider's language.
- **Voice:** transcribe a spoken "when's the next train to Berkeley?" with **Voxtral** and answer it from this index (and read it back with Voxtral TTS).

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly.

---
## ✅ Index loaded
`bart_realtime` and `bart_alerts` hold the latest snapshot (plus `muni_realtime` if you set a 511 token).

Re-run cells 3-4 to refresh.